Plan:
1. Connecting Google Drive and defining paths.
2. Loading the same 61 validation images as those used for Gemini.
3. Verifying the presence of images and masks.
4. Generating 61 comparison images featuring BiomedParse masks (already generated in notebook 5).

In [ ]:
# Import libraries and mount Google Drive.

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

from google.colab import drive
drive.mount("/content/drive")

# Define project directories.
PROJECT_DIR = Path("/content/drive/MyDrive/ovarian_ultrasound_vlm")

DATASET_DIR = Path(
    "/content/drive/MyDrive/biomedparse_project/"
    "nnunet_raw/Dataset001_Follicles"
)

IMAGES_DIR = DATASET_DIR / "imagesTr"
MASKS_DIR = DATASET_DIR / "labelsTr"

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

OUTPUT_DIR = PROJECT_DIR / "evaluation" / "biomedparse_qualitative"
FIGURES_DIR = PROJECT_DIR / "figures" / "biomedparse_qualitative"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Manifest exists:", MANIFEST_PATH.exists())
print("Images directory exists:", IMAGES_DIR.exists())
print("Masks directory exists:", MASKS_DIR.exists())

Mounted at /content/drive
Manifest exists: True
Images directory exists: True
Masks directory exists: True


This notebook evaluates the visual plausibility of BiomedParse pseudo-labels on the same 61 validation images used in the Gemini experiment

In [ ]:
# Define separate directories for Gemini and BiomedParse figures.

GEMINI_REVIEW_DIR = (
    PROJECT_DIR
    / "figures"
    / "semantic_evaluation"
    / "qualitative_review"
)

REVIEW_FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
    / "biomedparse_qualitative"
    / "qualitative_review"
)

# Prevent mixing the review figures of different models.
assert REVIEW_FIGURES_DIR.resolve() != GEMINI_REVIEW_DIR.resolve()

REVIEW_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Gemini figures:", GEMINI_REVIEW_DIR)
print("BiomedParse figures:", REVIEW_FIGURES_DIR)

Gemini figures: /content/drive/MyDrive/ovarian_ultrasound_vlm/figures/semantic_evaluation/qualitative_review
BiomedParse figures: /content/drive/MyDrive/ovarian_ultrasound_vlm/figures/biomedparse_qualitative/qualitative_review


Define a separate directory for BiomedParse figures.

In [ ]:
# Load the manifest and select validation images.

manifest = pd.read_csv(MANIFEST_PATH)
# Select the validation split using normalized labels.

split_values = manifest["split"].astype(str).str.strip().str.lower()

validation = manifest.loc[
    split_values.isin(["val", "validation"])
].copy()

print("Validation images:", len(validation))
print(validation["category"].value_counts())

assert len(validation) == 61, (
    f"Expected 61 validation images, found {len(validation)}"
)

print("Validation images:", len(validation))
print(validation["category"].value_counts())

assert len(validation) == 61
assert validation["case_id"].is_unique

# Verify image and pseudo-label paths.
missing_images = []
missing_masks = []

for row in validation.itertuples():
    if not Path(row.image_path).exists():
        missing_images.append(row.case_id)

    if not Path(row.pseudo_label_path).exists():
        missing_masks.append(row.case_id)

print("Missing images:", len(missing_images))
print("Missing masks:", len(missing_masks))

assert not missing_images, missing_images
assert not missing_masks, missing_masks

print("All 61 image-mask pairs are available.")

Validation images: 61
category
dominant    31
pco         21
normal       9
Name: count, dtype: int64
Validation images: 61
category
dominant    31
pco         21
normal       9
Name: count, dtype: int64
Missing images: 0
Missing masks: 0
All 61 image-mask pairs are available.


Use the same validation split as the Gemini evaluation to ensure a consistent comparison

In [ ]:
missing_images = []
missing_masks = []

for row in validation.itertuples():
    if not Path(row.image_path).exists():
        missing_images.append(row.case_id)

    if not Path(row.pseudo_label_path).exists():
        missing_masks.append(row.case_id)

print("Validation images:", len(validation))
print("Missing images:", len(missing_images))
print("Missing masks:", len(missing_masks))

assert len(validation) == 61
assert not missing_images, missing_images
assert not missing_masks, missing_masks

print("All validation image-mask pairs are available.")

Validation images: 61
Missing images: 0
Missing masks: 0
All validation image-mask pairs are available.


Verify all validation image-mask pairs

In [ ]:
# Generate side-by-side BiomedParse qualitative review figures.

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage import measure
from pathlib import Path

# Use a separate directory for comparison figures.
COMPARISON_DIR = (
    PROJECT_DIR
    / "figures"
    / "biomedparse_qualitative"
    / "qualitative_review_comparison"
)

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

# Check that the output directory is separate from Gemini.
assert COMPARISON_DIR.resolve() != GEMINI_REVIEW_DIR.resolve()

for _, row in validation.iterrows():

    case_id = row["case_id"]

    # Load the original ultrasound image.
    image = np.array(
        Image.open(row["image_path"]).convert("RGB")
    )

    # Load the BiomedParse pseudo-label mask.
    mask = np.array(
        Image.open(row["pseudo_label_path"])
    )

    assert mask.ndim == 2, case_id
    assert image.shape[:2] == mask.shape, case_id

    binary_mask = mask == 1

    # Count connected components using 8-connectivity.
    labeled_mask = measure.label(
        binary_mask,
        connectivity=2
    )
    region_count = labeled_mask.max()

    # Create a transparent blue overlay.
    overlay = np.zeros(
        (mask.shape[0], mask.shape[1], 4),
        dtype=float
    )

    overlay[binary_mask, :3] = [0.0, 0.5, 1.0]
    overlay[binary_mask, 3] = 0.30

    # Create a side-by-side comparison.
    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, 8)
    )

    # Original ultrasound.
    axes[0].imshow(image)
    axes[0].set_title(
        "Original Ultrasound",
        fontsize=14
    )
    axes[0].axis("off")

    # Ultrasound with BiomedParse pseudo-label.
    axes[1].imshow(image)
    axes[1].imshow(overlay)
    axes[1].set_title(
        "BiomedParse Pseudo-label Overlay",
        fontsize=14
    )
    axes[1].axis("off")

    fig.suptitle(
        f"{case_id} | BiomedParse count: {region_count}",
        fontsize=15
    )

    fig.subplots_adjust(
        left=0.02,
        right=0.98,
        bottom=0.04,
        top=0.90,
        wspace=0.06
    )

    output_path = (
        COMPARISON_DIR / f"{case_id}_review.png"
    )

    # Do not overwrite existing review figures.
    if not output_path.exists():
        fig.savefig(
            output_path,
            dpi=180,
            bbox_inches="tight"
        )

    plt.close(fig)

print("Comparison figures:", len(list(COMPARISON_DIR.glob("*_review.png"))))
print("Saved to:", COMPARISON_DIR)

Comparison figures: 61
Saved to: /content/drive/MyDrive/ovarian_ultrasound_vlm/figures/biomedparse_qualitative/qualitative_review_comparison


Generate large BiomedParse-only qualitative review figures

In [ ]:
# Create a BiomedParse review table using the Gemini review workflow.

import pandas as pd
import numpy as np
from PIL import Image
from skimage import measure
from pathlib import Path

review_records = []

for _, row in validation.iterrows():

    case_id = row["case_id"]

    # Load the BiomedParse pseudo-label mask.
    mask = np.array(
        Image.open(row["pseudo_label_path"])
    )

    assert mask.ndim == 2, f"Expected a 2D mask: {case_id}"

    # Identify connected components using 8-connectivity.
    binary_mask = mask == 1
    labeled_mask = measure.label(binary_mask, connectivity=2)
    regions = measure.regionprops(labeled_mask)

    # Include all region-level geometric measurements.
    region_descriptions = []

    for region_id, region in enumerate(regions, start=1):

        min_row, min_col, max_row, max_col = region.bbox

        region_descriptions.append(
            f"Region {region_id}: "
            f"area={region.area} px; "
            f"bbox=[{min_col}, {min_row}, {max_col}, {max_row}]"
        )

    review_records.append({
        "case_id": case_id,
        "category": row["category"],
        "biomedparse_count": len(regions),
        "region_descriptions": " | ".join(region_descriptions),
        "review_figure": str(
            COMPARISON_DIR / f"{case_id}_review.png"
            ),
        "overall_visual_assessment": "",
        "reviewer_notes": ""
    })

review_template = pd.DataFrame(review_records)

# Validate the review table.
assert len(review_template) == 61
assert review_template["case_id"].is_unique

assert review_template["review_figure"].apply(
    lambda path: Path(path).exists()
).all()

# Save the template without overwriting existing manual work.
REVIEW_CSV_PATH = (
    OUTPUT_DIR / "biomedparse_manual_review_template.csv"
)

if REVIEW_CSV_PATH.exists():
    print("Review template already exists; not overwritten.")
    print("Path:", REVIEW_CSV_PATH)
else:
    review_template.to_csv(
        REVIEW_CSV_PATH,
        index=False
    )
    print("Review template saved:", REVIEW_CSV_PATH)

print("Images for review:", len(review_template))

display(review_template.head())

Review template already exists; not overwritten.
Path: /content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/biomedparse_qualitative/biomedparse_manual_review_template.csv
Images for review: 61


,case_id,category,biomedparse_count,region_descriptions,review_figure,overall_visual_assessment,reviewer_notes
0,dominant_0011,dominant,1,"Region 1: area=16238.0 px; bbox=[279, 314, 428...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
1,dominant_0012,dominant,1,"Region 1: area=5733.0 px; bbox=[85, 305, 158, ...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
2,dominant_0017,dominant,1,"Region 1: area=16000.0 px; bbox=[93, 293, 262,...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
3,dominant_0019,dominant,1,"Region 1: area=7116.0 px; bbox=[214, 291, 313,...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
4,dominant_0021,dominant,1,"Region 1: area=26438.0 px; bbox=[81, 222, 272,...",/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,


Prepare a review table for the 61 validation images using the same assessment scale as Gemini: Good, Partial, Poor, and Unclear

In [ ]:
# Export the BiomedParse review table to a formatted Excel workbook.

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo

EXCEL_PATH = OUTPUT_DIR / "biomedparse_manual_review.xlsx"

if EXCEL_PATH.exists():
    print("Workbook already exists; not overwritten:", EXCEL_PATH)

else:
    wb = Workbook()
    ws = wb.active
    ws.title = "BiomedParse review"

    # Add column headers.
    ws.append(review_template.columns.tolist())

    # Add review records.
    for record in review_template.itertuples(index=False, name=None):
        ws.append(list(record))

    # Create a styled Excel table.
    table = Table(
        displayName="BiomedParseReview",
        ref=f"A1:G{ws.max_row}"
    )

    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2",
        showRowStripes=True,
        showColumnStripes=False
    )

    ws.add_table(table)

    # Configure column widths.
    column_widths = {
        "A": 22,  # case_id
        "B": 16,  # category
        "C": 22,  # biomedparse_count
        "D": 65,  # region_descriptions
        "E": 55,  # review_figure
        "F": 29,  # overall_visual_assessment
        "G": 85   # reviewer_notes
    }

    for column, width in column_widths.items():
        ws.column_dimensions[column].width = width

    # Format rows and enable text wrapping.
    ws.row_dimensions[1].height = 30

    for row in ws.iter_rows(min_row=2):
        ws.row_dimensions[row[0].row].height = 70

        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )

    # Add assessment dropdown.
    dropdown = DataValidation(
        type="list",
        formula1='"Good,Partial,Poor,Unclear"',
        allow_blank=True
    )

    dropdown.error = "Choose Good, Partial, Poor, or Unclear."
    dropdown.errorTitle = "Invalid assessment"
    dropdown.showErrorMessage = True

    ws.add_data_validation(dropdown)
    dropdown.add(f"F2:F{ws.max_row}")

    # Freeze header and identifier columns.
    ws.freeze_panes = "C2"

    wb.save(EXCEL_PATH)

    print("Formatted Excel workbook saved:", EXCEL_PATH)

Workbook already exists; not overwritten: /content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/biomedparse_qualitative/biomedparse_manual_review.xlsx


Making a table look nice

In [ ]:
# Load and validate the completed BiomedParse manual review.

review_path = OUTPUT_DIR / "biomedparse_manual_review.xlsx"

manual_review = pd.read_excel(
    review_path,
    engine="openpyxl",
    keep_default_na=False
)

valid_ratings = {"Good", "Partial", "Poor", "Unclear"}

manual_review["overall_visual_assessment"] = (
    manual_review["overall_visual_assessment"]
    .astype(str)
    .str.strip()
)

assert len(manual_review) == 61
assert manual_review["case_id"].is_unique
assert set(manual_review["case_id"]) == set(validation["case_id"])
assert manual_review["overall_visual_assessment"].isin(valid_ratings).all()

print("Reviewed images:", len(manual_review))
print("\nManual assessment:")
print(manual_review["overall_visual_assessment"].value_counts())

display(
    manual_review[
        ["case_id", "overall_visual_assessment", "reviewer_notes"]
    ].head()
)

Reviewed images: 61

Manual assessment:
overall_visual_assessment
Partial    29
Poor       17
Unclear     8
Good        7
Name: count, dtype: int64


,case_id,overall_visual_assessment,reviewer_notes
0,dominant_0011,Poor,BiomedParse marked one region. It´s hard to te...
1,dominant_0012,Poor,BiomedParse marked one region. It´s hard to te...
2,dominant_0017,Partial,I guess the blue region captures some of the f...
3,dominant_0019,Partial,BiomedParse captures one of the most obvious f...
4,dominant_0021,Partial,BiomedParse captures one of the most obvious f...


Load and validate the completed BiomedParse review for all 61 validation images

In [ ]:
# Calculate the final BiomedParse qualitative summary.

rating_order = ["Good", "Partial", "Poor", "Unclear"]

counts = (
    manual_review["overall_visual_assessment"]
    .value_counts()
    .reindex(rating_order, fill_value=0)
)

qualitative_summary = pd.DataFrame({
    "rating": rating_order,
    "count": counts.values,
    "percentage": (
        counts.values / len(manual_review) * 100
    ).round(2)
})

display(qualitative_summary)

summary_path = (
    OUTPUT_DIR / "biomedparse_qualitative_summary_final.csv"
)

# Protect an existing summary from accidental overwriting.
if summary_path.exists():
    print("Summary already exists:", summary_path)
else:
    qualitative_summary.to_csv(summary_path, index=False)
    print("Saved:", summary_path)

,rating,count,percentage
0,Good,7,11.48
1,Partial,29,47.54
2,Poor,17,27.87
3,Unclear,8,13.11


Saved: /content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/biomedparse_qualitative/biomedparse_qualitative_summary_final.csv


Calculate the distribution of visual assessment ratings and save the summary for comparison with Gemini